# 🧬 RAIM Scope — 양파 체세포 분열기 검출 모델 학습

**파이프라인:** RAIM Scope (라벨링) → COCO/YOLO 데이터셋 → **Colab (학습)** → ONNX → Hailo Compiler → Pi

## 5단계 검출 클래스
| ID | 영문 | 한국어 |
|----|------|--------|
| 0  | interphase | 간기 |
| 1  | prophase   | 전기 |
| 2  | metaphase  | 중기 |
| 3  | anaphase   | 후기 |
| 4  | telophase  | 말기 |

## 환경
- Google Colab (GPU 권장: T4 / A100)
- Ultralytics YOLOv8 (YOLOv8n — Hailo-10H 최적화 사이즈)
- 입력 데이터셋: RAIM Scope 데이터셋 탭에서 "Export YOLO" 로 만든 zip

## 사용 순서
1. Cell 2 실행 → 의존성 설치
2. Cell 3 실행 → 데이터셋 zip 업로드 (또는 Drive 마운트)
3. Cell 4 실행 → 데이터 검증
4. Cell 5 실행 → 학습 (100 epoch ≈ 30~60분)
5. Cell 6 실행 → 평가 (mAP, confusion matrix)
6. Cell 7 실행 → ONNX export (Hailo 입력)
7. Cell 8 → Hailo 변환 가이드 확인
8. Cell 9 실행 → 결과 다운로드

## Cell 2 — 환경 설정 (의존성 설치)

In [ ]:
# Ultralytics YOLOv8 + ONNX export
!pip install -q ultralytics onnx onnxruntime onnxslim

import torch
from ultralytics import YOLO
import os

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device  :', torch.cuda.get_device_name(0))
else:
    print('⚠️  GPU 가 감지되지 않았습니다. 런타임 → 런타임 유형 변경 → GPU(T4) 선택하세요.')

## Cell 3 — 데이터셋 마운트 (둘 중 하나 선택)

**옵션 A — Google Drive 마운트** (권장 — 재학습/저장 편리)  
**옵션 B — zip 직접 업로드** (1회성 빠른 학습)

In [ ]:
# === 옵션 A: Google Drive ===
# Drive 의 /MyDrive/RAIMScope/onion_dataset.zip 형태로 업로드해두면 재사용 가능
USE_DRIVE = True   # False 로 바꾸면 옵션 B 로
DRIVE_ZIP_PATH = '/content/drive/MyDrive/RAIMScope/onion_dataset.zip'
DATASET_DIR = '/content/onion_yolo'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p {DATASET_DIR}
    !unzip -oq "{DRIVE_ZIP_PATH}" -d {DATASET_DIR}
else:
    # === 옵션 B: 직접 업로드 ===
    from google.colab import files
    print('데이터셋 zip 파일을 업로드하세요 (RAIM Scope → 데이터셋 탭 → Export YOLO 결과)...')
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    !mkdir -p {DATASET_DIR}
    !unzip -oq "{zip_name}" -d {DATASET_DIR}

# data.yaml 위치 자동 탐지 (zip 구조에 따라 1단계 깊이 다를 수 있음)
import glob
yaml_candidates = glob.glob(f'{DATASET_DIR}/**/data.yaml', recursive=True)
assert yaml_candidates, f'❌ data.yaml 을 찾을 수 없습니다. 위치 확인: {DATASET_DIR}'
DATA_YAML = yaml_candidates[0]
DATA_ROOT = os.path.dirname(DATA_YAML)
print('✅ data.yaml :', DATA_YAML)
print('✅ dataset   :', DATA_ROOT)

## Cell 4 — 데이터 검증 (클래스 카운트 + 샘플 시각화)

In [ ]:
import yaml, glob, os, random
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

with open(DATA_YAML, 'r') as f:
    data_cfg = yaml.safe_load(f)

print('📋 data.yaml 내용')
print('  path :', data_cfg.get('path'))
print('  train:', data_cfg.get('train'))
print('  val  :', data_cfg.get('val'))
print('  nc   :', data_cfg.get('nc'))
print('  names:', data_cfg.get('names'))

# data.yaml 의 path 가 절대경로면 그대로, 상대면 DATA_ROOT 기준으로 보정
if not os.path.isabs(data_cfg.get('path', '')) or not os.path.isdir(data_cfg.get('path', '')):
    data_cfg['path'] = DATA_ROOT
    with open(DATA_YAML, 'w') as f:
        yaml.safe_dump(data_cfg, f, allow_unicode=True)
    print('🔧 data.yaml path 자동 수정 →', DATA_ROOT)

# 클래스별 라벨 카운트
names = data_cfg['names']
if isinstance(names, dict):
    names = [names[i] for i in sorted(names.keys())]

for split in ('train', 'val'):
    label_dir = os.path.join(DATA_ROOT, 'labels', split)
    if not os.path.isdir(label_dir):
        continue
    counter = Counter()
    n_imgs = 0
    for txt in glob.glob(os.path.join(label_dir, '*.txt')):
        n_imgs += 1
        with open(txt) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1
    print(f'\n📊 [{split}] 이미지 {n_imgs}장')
    for cid in range(len(names)):
        print(f'   {cid} {names[cid]:<12s}: {counter.get(cid, 0):>4d}')

# 학습 데이터 샘플 4장 시각화
img_train = glob.glob(os.path.join(DATA_ROOT, 'images/train', '*.*'))
if img_train:
    samples = random.sample(img_train, min(4, len(img_train)))
    fig, axes = plt.subplots(1, len(samples), figsize=(4*len(samples), 4))
    if len(samples) == 1:
        axes = [axes]
    for ax, ip in zip(axes, samples):
        img = Image.open(ip).convert('RGB')
        W, H = img.size
        draw = ImageDraw.Draw(img)
        lp = ip.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        if os.path.exists(lp):
            with open(lp) as f:
                for line in f:
                    p = line.strip().split()
                    if len(p) == 5:
                        cid, xc, yc, w, h = int(p[0]), float(p[1]), float(p[2]), float(p[3]), float(p[4])
                        x1 = (xc - w/2)*W; y1 = (yc - h/2)*H
                        x2 = (xc + w/2)*W; y2 = (yc + h/2)*H
                        draw.rectangle([x1, y1, x2, y2], outline='red', width=3)
                        draw.text((x1+3, y1+3), names[cid], fill='red')
        ax.imshow(img)
        ax.set_title(os.path.basename(ip))
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## Cell 5 — 학습 (YOLOv8n, 100 epochs)

**기본 설정** (T4 기준 약 30~60분 소요)
- 모델: YOLOv8n (가장 가벼운 nano — Hailo-10H 권장)
- 입력 크기: 640×640
- Batch: 16 (메모리 부족하면 8 로 낮추기)
- Epochs: 100 (조기 종료 patience=20)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')   # ImageNet pretrained — 전이학습

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project='runs/onion',
    name='yolov8n_v1',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    augment=True,
    mixup=0.1,
    mosaic=1.0,
)

RUN_DIR = 'runs/onion/yolov8n_v1'
BEST_PT = os.path.join(RUN_DIR, 'weights/best.pt')
print('✅ 학습 완료 — best 가중치:', BEST_PT)

## Cell 6 — 결과 평가 (mAP, confusion matrix)

In [ ]:
from IPython.display import Image as IPyImage, display

# best 모델로 검증 데이터 평가
best = YOLO(BEST_PT)
metrics = best.val(data=DATA_YAML, imgsz=640, plots=True)

print('\n📈 평가 결과')
print(f'  mAP50      : {metrics.box.map50:.4f}')
print(f'  mAP50-95   : {metrics.box.map:.4f}')
print(f'  Precision  : {metrics.box.mp:.4f}')
print(f'  Recall     : {metrics.box.mr:.4f}')

# 자동 생성된 그래프 표시 — 학습곡선/confusion matrix
for plot_name in ['results.png', 'confusion_matrix.png',
                  'PR_curve.png', 'F1_curve.png']:
    plot_path = os.path.join(RUN_DIR, plot_name)
    if os.path.exists(plot_path):
        print(f'\n--- {plot_name} ---')
        display(IPyImage(plot_path))

## Cell 7 — ONNX Export (Hailo 컴파일러 입력)

Hailo-10H 는 **opset=11**, **고정 입력 크기**, **dynamic axes 비활성** 의 ONNX 를 요구합니다.

In [ ]:
# Hailo Compiler 호환 ONNX export
best = YOLO(BEST_PT)
onnx_path = best.export(
    format='onnx',
    imgsz=640,
    opset=11,
    simplify=True,
    dynamic=False,
)
print('✅ ONNX export :', onnx_path)

# 출력 확인
import onnx
m = onnx.load(onnx_path)
print('  Opset       :', m.opset_import[0].version)
print('  Inputs      :', [(i.name, [d.dim_value for d in i.type.tensor_type.shape.dim]) for i in m.graph.input])
print('  Outputs     :', [(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim]) for o in m.graph.output])

## Cell 8 — Hailo 변환 가이드 (로컬 PC 작업)

ONNX → HEF (Hailo Executable Format) 변환은 **Hailo Dataflow Compiler (DFC)** 가 필요합니다.
Colab 에선 라이선스 문제로 실행할 수 없으므로 다운받은 onnx 를 로컬에서 변환합니다.

### 1. Hailo SW Suite 설치 (Linux x86_64 / WSL2 / 우분투)
```bash
# https://hailo.ai/developer-zone/ 에서 SW Suite Docker 이미지 다운
docker pull hailo_ai_sw_suite:latest
docker run -it --rm -v $(pwd):/workspace hailo_ai_sw_suite:latest bash
```

### 2. ONNX 파싱 + 캘리브레이션
```bash
# 캘리브레이션용 학습 이미지 200~500장 준비 (images/train/ 활용)
hailo parser onnx best.onnx --hw-arch hailo10h --har-output best.har

hailo optimize best.har \
    --hw-arch hailo10h \
    --calib-set-path /workspace/calib_imgs/ \
    --output-har-path best_optimized.har
```

### 3. HEF 컴파일 (NMS-postprocess 활성화)
```bash
# RAIM Scope HailoYOLOInference 가 NMS 출력을 기대 → 반드시 NMS 포함
hailo compiler best_optimized.har \
    --hw-arch hailo10h \
    --output-dir ./compiled \
    --hef-name onion_mitosis.hef
```

### 4. Pi 에 배포
```bash
# 라즈베리파이 5 + Hailo-10H 모듈
scp compiled/onion_mitosis.hef neomscope@192.168.123.108:~/workspace/NeoScope_dev/models/
```
RAIM Scope GUI → **검출 탭** → 모델 드롭다운에서 `onion_mitosis.hef` 선택 → **AI 활성화**.

### 5. 빠른 검증 (HailoRT runtime 만 — Pi 에서)
```bash
hailortcli benchmark onion_mitosis.hef
# 권장 FPS: 30+ @ Pi5 + Hailo-10H
```

## Cell 9 — 결과 다운로드 (best.pt + best.onnx)

In [ ]:
from google.colab import files
import shutil

# 패키징: best.pt + best.onnx + 그래프 + data.yaml
OUT_DIR = '/content/onion_release'
os.makedirs(OUT_DIR, exist_ok=True)

shutil.copy(BEST_PT, OUT_DIR)
shutil.copy(onnx_path, OUT_DIR)
shutil.copy(DATA_YAML, OUT_DIR)
for plot_name in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    p = os.path.join(RUN_DIR, plot_name)
    if os.path.exists(p):
        shutil.copy(p, OUT_DIR)

# zip + 다운로드
zip_path = '/content/onion_mitosis_release.zip'
shutil.make_archive(zip_path[:-4], 'zip', OUT_DIR)
print('✅ 패키지 :', zip_path)
print('  포함   : best.pt, best.onnx, data.yaml, 학습 그래프')

# Drive 백업 (선택)
if USE_DRIVE:
    drive_out = '/content/drive/MyDrive/RAIMScope/onion_mitosis_release.zip'
    shutil.copy(zip_path, drive_out)
    print('☁️  Drive 백업 :', drive_out)

files.download(zip_path)